In [1]:


import mlflow
# Step 2: Set up the MLflow tracking server
mlflow.set_tracking_uri("http://ec2-13-62-229-17.eu-north-1.compute.amazonaws.com:5000/")

In [2]:
# set or create an experiment
mlflow.set_experiment("exp7_sbert_lightgbm_optuna_hpt") 


2025/12/01 23:33:14 INFO mlflow.tracking.fluent: Experiment with name 'exp7_sbert_lightgbm_optuna_hpt' does not exist. Creating a new experiment.


<Experiment: artifact_location='s3://my-s3-bucket-of-store-artifact-youtube-data12/mlflow-artifacts/9', creation_time=1764612194694, experiment_id='9', last_update_time=1764612194694, lifecycle_stage='active', name='exp7_sbert_lightgbm_optuna_hpt', tags={}>

In [ ]:
import pandas as pd
df=pd.read_csv('sentiment_clean.csv')

In [11]:

df['sentiment_numeric']=df.pop('sentiment_numeric')


In [ ]:

import os
import numpy as np
import joblib
import mlflow
import optuna
import lightgbm as lgb

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, recall_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

# -------------------------
# CONFIG
# -------------------------
RANDOM_STATE = 42
TEST_SIZE = 0.20
N_FOLDS = 3                        # inner CV for Optuna (keeps tuning time reasonable)
N_TRIALS = 50                      # adjust: 30-100 depending on time
EARLY_STOPPING_ROUNDS = 50
MLFLOW_EXPERIMENT_NAME = "Exp7_SBERT_LightGBM"
SBERT_MODEL = "all-MiniLM-L6-v2"   # fast & high quality
TF_BATCH_SIZE = 256                # embedding batch size
N_JOBS = -1



# Map target to integers consistently
df['sentiment_numeric'] = df['sentiment_numeric'].map({-1: 2, 0: 0, 1: 1})
df = df.dropna(subset=['text_clean', 'sentiment_numeric']).reset_index(drop=True)
y = df['sentiment_numeric'].astype(int)

# (Optional) numeric features - if you have numeric columns put them here; otherwise skip
# Here I assume the same slicing you used earlier (all columns except first and last are numeric)
if df.shape[1] > 2:
    X_numeric = df.iloc[:, 1:-1]
    scaler = StandardScaler(with_mean=False)
    X_numeric_scaled = scaler.fit_transform(X_numeric)
    has_numeric = True
else:
    X_numeric_scaled = None
    has_numeric = False

# -------------------------
# Create SBERT embeddings
# -------------------------
print("Loading SBERT model:", SBERT_MODEL)
sbert = SentenceTransformer(SBERT_MODEL)

texts = df['text_clean'].astype(str).tolist()
embeddings = []
print("Creating SBERT embeddings (batches):")
for i in tqdm(range(0, len(texts), TF_BATCH_SIZE)):
    batch = texts[i:i+TF_BATCH_SIZE]
    emb = sbert.encode(batch, show_progress_bar=False, convert_to_numpy=True)
    embeddings.append(emb)
embeddings = np.vstack(embeddings)   # shape: (n_samples, emb_dim)
print("Embeddings shape:", embeddings.shape)

# Combine text embeddings + numeric features (if any)
if has_numeric:
    from scipy import sparse
    X_numeric_arr = np.asarray(X_numeric_scaled)
    X = np.hstack([embeddings, X_numeric_arr])
else:
    X = embeddings

# -------------------------
# Train/test split (stratified)
# -------------------------
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

# compute sample/class weights on training set
classes = np.unique(y_train_full)
cw = compute_class_weight(class_weight="balanced", classes=classes, y=y_train_full)
class_weight_dict = {int(c): float(w) for c, w in zip(classes, cw)}
sample_weight_full = np.array([class_weight_dict[int(lbl)] for lbl in y_train_full])

print("Class weights:", class_weight_dict)

# -------------------------
# Optuna objective (uses StratifiedKFold CV)
# maximize macro recall
# -------------------------
def objective(trial):
    params = {
        "boosting_type": "gbdt",
        "objective": "multiclass",
        "num_class": len(classes),
        "metric": "multi_logloss",
        "n_jobs": N_JOBS,
        "verbosity": -1,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 31, 256),
        "max_depth": trial.suggest_int("max_depth", 3, 16),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 200),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.5, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.5, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 0, 7),
        "lambda_l1": trial.suggest_float("lambda_l1", 0.0, 2.0),
        "lambda_l2": trial.suggest_float("lambda_l2", 0.0, 2.0),
        # keep class_weight as dictionary so LightGBM can use it
        "class_weight": class_weight_dict,
        "n_estimators": trial.suggest_int("n_estimators", 200, 1000),
        "random_state": RANDOM_STATE
    }

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    recalls = []

    # use stratified folds and early stopping on fold validation
    for train_idx, val_idx in skf.split(X_train_full, y_train_full):
        X_tr, X_val = X_train_full[train_idx], X_train_full[val_idx]
        y_tr, y_val = y_train_full.iloc[train_idx], y_train_full.iloc[val_idx]

        sw_tr = np.array([class_weight_dict[int(lbl)] for lbl in y_tr])

        model = lgb.LGBMClassifier(**params)
        model.fit(
            X_tr, y_tr,
            sample_weight=sw_tr,
            eval_set=[(X_val, y_val)],
            eval_metric="multi_logloss",
            callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS), lgb.log_evaluation(0)]
        )

        preds = model.predict(X_val)
        r = recall_score(y_val, preds, average="macro")
        recalls.append(r)

    # return mean macro recall across folds
    return float(np.mean(recalls))

# run study
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print("Best optuna params:", study.best_params)

# -------------------------
# Final train on full train set with best params
# -------------------------
best_params = study.best_params.copy()
best_params.update({
    "boosting_type": "gbdt",
    "objective": "multiclass",
    "num_class": len(classes),
    "metric": "multi_logloss",
    "class_weight": class_weight_dict,
    "random_state": RANDOM_STATE,
    "n_jobs": N_JOBS,
    "verbosity": -1
})

final_model = lgb.LGBMClassifier(**best_params)
final_model.fit(
    X_train_full, y_train_full,
    sample_weight=sample_weight_full,
    eval_set=[(X_test, y_test)],
    eval_metric="multi_logloss",
    callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS)]
)

# -------------------------
# Evaluate
# -------------------------
y_pred = final_model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
macro_rec = recall_score(y_test, y_pred, average="macro")
report = classification_report(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print("\nFINAL RESULTS")
print("Accuracy:", acc)
print("Macro Recall:", macro_rec)
print(report)
print("Confusion Matrix:\n", cm)

# -------------------------
# MLflow logging & saving artifacts
# -------------------------
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
with mlflow.start_run(run_name="Exp7_SBERT_LGBM"):
    mlflow.log_params(best_params)
    mlflow.log_metric("accuracy", float(acc))
    mlflow.log_metric("macro_recall", float(macro_rec))

    # class-wise metrics to MLflow
    rpt_dict = lgb.scikit_learn._check_classification_report(y_test, y_pred) if False else None
    # (we already printed report; you can parse and log individual metrics if desired)

    # save confusion matrix image
    import matplotlib.pyplot as plt
    import seaborn as sns
    plt.figure(figsize=(7,6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title("Confusion Matrix -  SBERT+LGBM")
    plt.savefig("exp7_confusion_matrix.png")
    mlflow.log_artifact("exp7_confusion_matrix.png")
    plt.close()

    # save model + sbert + scaler
    joblib.dump(final_model, "exp7_lgbm_model.pkl")
    joblib.dump(sbert, "exp7_sbert_model.pkl")        # sentence-transformer object
    if has_numeric:
        joblib.dump(scaler, "exp7_numeric_scaler.pkl")
    mlflow.log_artifact("exp7_lgbm_model.pkl")
    mlflow.log_artifact("exp7_sbert_model.pkl")
    if has_numeric:
        mlflow.log_artifact("exp7_numeric_scaler.pkl")

print("Artifacts saved. Done.")


Loading SBERT model: all-MiniLM-L6-v2
Creating SBERT embeddings (batches):


  0%|          | 0/194 [00:00<?, ?it/s]

Embeddings shape: (49477, 384)


[I 2025-12-02 00:02:10,531] A new study created in memory with name: no-name-c699595b-7a12-4d03-901f-e9d4c8ef8338


Class weights: {0: 0.7256444102225644, 1: 0.7350641632774342, 2: 3.8242512077294686}


...

[I 2025-12-02 03:39:59,032] Trial 49 finished with value: 0.7378752505379812 and parameters: {'learning_rate': 0.06914044506427788, 'num_leaves': 87, 'max_depth': 6, 'min_child_samples': 84, 'feature_fraction': 0.9013619759584941, 'bagging_fraction': 0.7853753836143081, 'bagging_freq': 2, 'lambda_l1': 0.6952536660646795, 'lambda_l2': 1.1133288632459288, 'n_estimators': 446}. Best is trial 39 with value: 0.7446808789723939.
Best optuna params: {'learning_rate': 0.039968533424280965, 'num_leaves': 139, 'max_depth': 7, 'min_child_samples': 191, 'feature_fraction': 0.8292098515690053, 'bagging_fraction': 0.8112319335914873, 'bagging_freq': 2, 'lambda_l1': 1.7051480703879394, 'lambda_l2': 0.807323091270011, 'n_estimators': 515}
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[515]	valid_0's multi_logloss: 0.593636


e:\mlops_projects\influence_mirror\myenv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



FINAL RESULTS
Accuracy: 0.7389854486661277
Macro Recall: 0.7548486735009144
              precision    recall  f1-score   support

           0       0.74      0.75      0.75      4546
           1       0.82      0.71      0.76      4488
           2       0.51      0.80      0.62       862

    accuracy                           0.74      9896
   macro avg       0.69      0.75      0.71      9896
weighted avg       0.76      0.74      0.74      9896

Confusion Matrix:
 [[3432  642  472]
 [1103 3193  192]
 [ 118   56  688]]


2025/12/02 03:41:46 INFO mlflow.tracking.fluent: Experiment with name 'Exp7_SBERT_LightGBM' does not exist. Creating a new experiment.


🏃 View run Exp7_SBERT_LGBM at: http://ec2-13-62-229-17.eu-north-1.compute.amazonaws.com:5000/#/experiments/10/runs/bf2eeb5477804614be0c7ce17c2d33f7
🧪 View experiment at: http://ec2-13-62-229-17.eu-north-1.compute.amazonaws.com:5000/#/experiments/10
Artifacts saved. Done.


In [9]:
! pip install -U sentence-transformers



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip
